## **Exploratory Analysis: Clustering for a New Labeling Strategy**
This section will guide you through the initial steps of preparing your data for unsupervised learning (clustering) to identify patterns that could inform a new labeling strategy.

### **1. Setup and Imports**
First, we'll import the necessary libraries and configure logging.

In [5]:
# --- Imports ---
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA # Import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from typing import Dict, Any, List, Optional, Tuple

# Add project root to Python path (assuming this notebook is run from project root or scripts folder)
PROJECT_ROOT = Path(__file__).resolve().parent.parent if '__file__' in locals() else Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables (if you use a .env file for paths/configs)
from dotenv import load_dotenv
load_dotenv()

# Import configuration and utilities
try:
    from config.paths import PATHS
    from config.params import GENERAL_CONFIG, LABELING_CONFIG, STRATEGY_CONFIG
    from utils.data_manager import DataManager
except ImportError as e:
    print(f"CRITICAL ERROR: Failed to import necessary modules. Ensure your project structure and dependencies are correct. Error: {e}")
    sys.exit(1) # Exit if core imports fail

print("Starting exploratory analysis for clustering-based labeling strategy.")

# --- Define parameters for data loading (adjust as needed) ---
SYMBOL = 'ADAUSDT' # Example symbol
INTERVAL = '5m'    # Example interval

# Define a forward window for calculating future returns for cluster interpretation
# This should be a reasonable period over which you expect a price movement to materialize.
# You can adjust this later based on your strategy's time horizon.
FUTURE_RETURN_WINDOW = 150 # bars

# --- PCA Parameters (New) ---
# You can adjust these based on your exploratory analysis
PCA_N_COMPONENTS = 0.95 # float (0-1) for variance explained, or int for number of components
                        # 0.95 means retain 95% of variance.

Starting exploratory analysis for clustering-based labeling strategy.


### **2. Load Processed Data**
We'll use your DataManager to load the processed data, which should contain all your engineered features along with OHLCV data.

In [3]:
# --- Load Processed Data ---
dm = DataManager()

try:
    df_processed = dm.load_data(
        symbol=SYMBOL,
        interval=INTERVAL,
        data_type='processed'
    )
    print(f"Successfully loaded processed data for {SYMBOL} {INTERVAL}. Shape: {df_processed.shape}")
    print(f"\nDataFrame Head:\n{df_processed.head()}")
    print(f"\nDataFrame Info:\n")
    df_processed.info()
    print(f"\nDataFrame Description:\n{df_processed.describe()}")

except FileNotFoundError:
    print(f"CRITICAL ERROR: Processed data file not found for {SYMBOL} {INTERVAL}. "
          f"Please ensure you have run the feature engineering script first.")
    sys.exit(1)
except Exception as e:
    print(f"CRITICAL ERROR: Error loading processed data for {SYMBOL} {INTERVAL}: {e}")
    sys.exit(1)

# Basic validation
if df_processed.empty:
    print("CRITICAL ERROR: Loaded processed data is empty. Cannot proceed with analysis.")
    sys.exit(1)
if not isinstance(df_processed.index, pd.DatetimeIndex):
    print("CRITICAL ERROR: Loaded DataFrame does not have a DatetimeIndex. Please ensure your data processing pipeline sets the index correctly.")
    sys.exit(1)
if not all(col in df_processed.columns for col in ['open', 'high', 'low', 'close', 'volume']):
    print("CRITICAL ERROR: Loaded DataFrame is missing essential OHLCV columns (open, high, low, close, volume).")
    sys.exit(1)

print("Initial data loading and inspection complete.")

2025-07-29 12:07:35,118 - INFO - Attempting to load data (DataFrame) from c:\Users\dimit\Downloads\CryptoFuturesBot\data\processed\ADAUSDT_5m_processed.parquet
2025-07-29 12:07:35,939 - INFO - Successfully loaded data from c:\Users\dimit\Downloads\CryptoFuturesBot\data\processed\ADAUSDT_5m_processed.parquet. Shape: (139180, 126)


Successfully loaded processed data for ADAUSDT 5m. Shape: (139180, 126)

DataFrame Head:
                             open    high     low   close   volume  \
timestamp                                                            
2024-01-01 00:00:00+00:00  0.5941  0.5975  0.5932  0.5973  3778102   
2024-01-01 00:05:00+00:00  0.5972  0.5973  0.5958  0.5967  1656258   
2024-01-01 00:10:00+00:00  0.5967  0.5987  0.5964  0.5974  1899024   
2024-01-01 00:15:00+00:00  0.5974  0.5984  0.5960  0.5961  1924883   
2024-01-01 00:20:00+00:00  0.5961  0.5971  0.5954  0.5966  1470259   

                           log_returns  typical_price     atr_5  atr_14  \
timestamp                                                                 
2024-01-01 00:00:00+00:00          NaN            NaN  0.000000     0.0   
2024-01-01 00:05:00+00:00          NaN       0.596000  0.000000     0.0   
2024-01-01 00:10:00+00:00    -0.001005       0.596600  0.000000     0.0   
2024-01-01 00:15:00+00:00     0.001172       

### **3. Feature Selection and Preprocessing for Clustering**

In [6]:
# --- Feature Selection for Clustering (Simplified with PCA) ---

# Get all numeric columns that are NOT part of the raw OHLCV data.
# This is a robust way to select all engineered features.
EXCLUDE_COLS_FOR_FEATURES = ['open', 'high', 'low', 'close', 'volume', 'open_time', 'vol_adj', 'label']
# Also exclude any non-numeric columns that might have been added by feature engineer (e.g., 'fvg', 'volatility_regime' if they are not numeric yet)
# We will convert them to numeric if possible or exclude them.
# Let's get all numeric columns first and then filter out the OHLCV ones.
all_numeric_cols = df_processed.select_dtypes(include=np.number).columns.tolist()
FEATURES_FOR_CLUSTERING = [col for col in all_numeric_cols if col not in EXCLUDE_COLS_FOR_FEATURES]

if not FEATURES_FOR_CLUSTERING:
    print("CRITICAL ERROR: No numeric features identified for clustering after excluding OHLCV and other known columns. Please check your data.")
    sys.exit(1)

print(f"Identified {len(FEATURES_FOR_CLUSTERING)} potential features for clustering.")
# print(f"Selected features: {FEATURES_FOR_CLUSTERING[:5]}... (showing first 5)") # Optional: print a subset

# --- Handle NaNs in Features ---
# Drop rows that have NaNs in any of the selected features.
initial_rows_features = len(df_processed)
df_features_for_pca = df_processed[FEATURES_FOR_CLUSTERING].copy()
df_features_for_pca.dropna(inplace=True) # Drop rows with NaNs in features
df_aligned_for_clustering = df_processed.loc[df_features_for_pca.index].copy() # Align original df to cleaned features

rows_dropped_features = initial_rows_features - len(df_features_for_pca)
if rows_dropped_features > 0:
    print(f"WARNING: Dropped {rows_dropped_features} rows due to NaNs in selected features for clustering.")
if df_features_for_pca.empty:
    print("CRITICAL ERROR: DataFrame for PCA is empty after dropping NaNs. Cannot proceed.")
    sys.exit(1)

print(f"Cleaned feature data shape for PCA: {df_features_for_pca.shape}")

# --- Feature Scaling (Pre-PCA) ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features_for_pca)
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES_FOR_CLUSTERING, index=df_features_for_pca.index)
print(f"Features scaled using StandardScaler. Scaled data shape: {X_scaled_df.shape}")

# --- Apply PCA for Dimensionality Reduction ---
pca = PCA(n_components=PCA_N_COMPONENTS, random_state=GENERAL_CONFIG.get('random_seed', 42))
X_pca = pca.fit_transform(X_scaled_df)

# Convert PCA results back to DataFrame for easier handling
# The column names will be 'PC1', 'PC2', etc.
pca_component_names = [f'PC{i+1}' for i in range(X_pca.shape[1])]
X_pca_df = pd.DataFrame(X_pca, columns=pca_component_names, index=X_scaled_df.index)

print(f"PCA applied. Reduced data shape: {X_pca_df.shape}")
print(f"Explained variance ratio by components: {pca.explained_variance_ratio_.sum():.4f}")
print(f"Number of components selected by PCA: {pca.n_components_}")
print(f"\nPCA Features Head:\n{X_pca_df.head()}")

# --- Calculate Future Returns for Cluster Interpretation ---
# We use the 'close' price from the original, aligned DataFrame (df_aligned_for_clustering).
df_aligned_for_clustering['future_close'] = df_aligned_for_clustering['close'].shift(-FUTURE_RETURN_WINDOW)
df_aligned_for_clustering['future_return_pct'] = ((df_aligned_for_clustering['future_close'] - df_aligned_for_clustering['close']) / df_aligned_for_clustering['close']) * 100

# Drop NaNs introduced by the shift for future return calculation
df_aligned_for_clustering.dropna(subset=['future_return_pct'], inplace=True)

# Align X_pca_df with the future_return_pct DataFrame's index
common_index_for_analysis = X_pca_df.index.intersection(df_aligned_for_clustering.index)
X_for_clustering_final = X_pca_df.loc[common_index_for_analysis].copy() # This is our final feature set for clustering
future_returns_final = df_aligned_for_clustering.loc[common_index_for_analysis, 'future_return_pct'].copy()

if X_for_clustering_final.empty or future_returns_final.empty:
    print("CRITICAL ERROR: Data for clustering and future return calculation is empty after alignment and NaN removal. Adjust FUTURE_RETURN_WINDOW or check data.")
    sys.exit(1)

print(f"Final data shape for clustering and future return analysis: {X_for_clustering_final.shape}")
print(f"Future returns calculated over {FUTURE_RETURN_WINDOW} bars.")
print(f"\nFuture Returns Head (aligned to PCA features):\n{future_returns_final.head()}")

Identified 121 potential features for clustering.
Cleaned feature data shape for PCA: (138892, 121)
Features scaled using StandardScaler. Scaled data shape: (138892, 121)
PCA applied. Reduced data shape: (138892, 34)
Explained variance ratio by components: 0.9541
Number of components selected by PCA: 34

PCA Features Head:
                                PC1       PC2       PC3       PC4       PC5  \
timestamp                                                                     
2024-01-02 00:00:00+00:00 -0.464235  7.819294 -0.919902 -1.993416 -0.001999   
2024-01-02 00:05:00+00:00 -0.260236  6.730857 -0.177853 -1.883549  0.577197   
2024-01-02 00:10:00+00:00 -0.231726  6.702399 -0.079148 -1.840789  0.709944   
2024-01-02 00:15:00+00:00 -0.210688  6.336932  0.177935 -1.819601  0.581418   
2024-01-02 00:20:00+00:00 -0.133725  6.711821  0.055808 -1.647768  0.156184   

                                PC6       PC7       PC8       PC9      PC10  \
timestamp                                 